In [14]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# plotting 설정 (누락 방지)
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

from datetime import datetime, timedelta
from typing import Iterable

from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import requests

# 사용자 유틸 (fetch_table_data, merge_endog_exog_data, get_market_cap, get_db_host 등)
from DATA.stock_invest_function import *
from datetime import datetime, timedelta
# SQLAlchemy
from sqlalchemy import create_engine, text, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

# -----------------------------
# Parameters
# -----------------------------
tic_name = 'ANET'                # 티커
hs_code = '851762'             # 대외 변수 HS CODE (옵션)
item_name = 'PSR'
st_date = '2010-01-01'
end_date = '2025-07-31'
today_date = pd.to_datetime(datetime.today().date())

USE_EXOGENOUS = True             # 외생변수 사용 여부

apikey = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# -----------------------------
# Forecasting Functions
# -----------------------------
def forecast_psr_with_lstm(data, forecast_steps=12, lookback_window=12):
    """PSR forecasting using LSTM (optional)"""
    try:
        from sklearn.preprocessing import MinMaxScaler
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from tensorflow.keras.optimizers import Adam
    except ImportError:
        print("TensorFlow not available, skipping LSTM forecasting")
        return None

    df = data.copy().sort_values('date').reset_index(drop=True)

    if len(df) < lookback_window + 1:
        print("Insufficient data for LSTM")
        return None

    values = df['endog_var'].values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(values)

    X, y = [], []
    for i in range(lookback_window, len(scaled_data)):
        X.append(scaled_data[i - lookback_window:i, 0])
        y.append(scaled_data[i, 0])

    if len(X) == 0:
        return None

    X, y = np.array(X), np.array(y)
    X = np.reshape(X, (X.shape[0], X.shape[1], 1))

    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=(lookback_window, 1)))
    model.add(Dropout(0.2))
    model.add(LSTM(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(25))
    model.add(Dense(1))

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    model.fit(X, y, batch_size=1, epochs=20, verbose=0)

    last_sequence = scaled_data[-lookback_window:].reshape(1, lookback_window, 1)
    predictions = []

    for _ in range(forecast_steps):
        pred = model.predict(last_sequence, verbose=0)
        predictions.append(pred[0, 0])
        last_sequence = np.roll(last_sequence, -1, axis=1)
        last_sequence[0, -1, 0] = pred[0, 0]

    predictions = np.array(predictions).reshape(-1, 1)
    predictions = scaler.inverse_transform(predictions)

    last_date = pd.to_datetime(df['date'].iloc[-1])
    forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                                   periods=forecast_steps, freq='M')

    return pd.DataFrame({'date': forecast_dates, 'forecasted_PSR': predictions.flatten()})


def forecast_psr_with_prophet(data, forecast_steps=12):
    """PSR forecasting using Prophet (optional)"""
    try:
        from prophet import Prophet
    except ImportError:
        print("Prophet not available, skipping Prophet forecasting")
        return None

    df_prophet = data[['date', 'endog_var']].copy()
    df_prophet = df_prophet.rename(columns={'date': 'ds', 'endog_var': 'y'})
    df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])

    if len(df_prophet) < 10:
        print("Insufficient data for Prophet")
        return None

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05
    )

    try:
        model.fit(df_prophet)
        future = model.make_future_dataframe(periods=forecast_steps, freq='M')
        forecast = model.predict(future)
        forecast_final = forecast.tail(forecast_steps)[['ds', 'yhat']].copy()
        forecast_final = forecast_final.rename(columns={'ds': 'date', 'yhat': 'forecasted_PSR'})
        forecast_final['forecasted_PSR'] = forecast_final['forecasted_PSR'].clip(lower=0.1)
        return forecast_final
    except Exception as e:
        print(f"Prophet forecasting failed: {e}")
        return None


def enhanced_sarima_forecast(data, exog_col=None, forecast_steps=4, use_log=False):
    """Enhanced SARIMA forecasting (quarterly/monthly 공용)"""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from itertools import product
    import warnings
    warnings.filterwarnings('ignore')

    df = data.copy().sort_values('date')
    endog = df['endog_var']
    exog = df[exog_col] if exog_col and exog_col in df.columns else None

    if use_log:
        endog = np.log(endog)

    p_values = [0, 1, 2]
    d_values = [0, 1]
    q_values = [0, 1, 2]
    P_values = [0, 1]
    D_values = [0, 1]
    Q_values = [0, 1]
    s_value = 4  # 분기 계절

    best_aic = float('inf')
    best_params = None
    best_model = None

    for p, d, q, P, D, Q in product(p_values, d_values, q_values, P_values, D_values, Q_values):
        try:
            total_params = p + q + P + Q + 1
            if total_params >= len(endog) * 0.3:
                continue
            model = SARIMAX(
                endog, exog=exog, order=(p, d, q),
                seasonal_order=(P, D, Q, s_value),
                enforce_stationarity=False, enforce_invertibility=False
            )
            fitted_model = model.fit(disp=False, maxiter=100)
            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_params = (p, d, q, P, D, Q, s_value)
                best_model = fitted_model
        except Exception:
            continue

    if best_model is None:
        try:
            model = SARIMAX(endog, exog=exog, order=(1, 1, 1),
                            seasonal_order=(0, 0, 0, 0),
                            enforce_stationarity=False, enforce_invertibility=False)
            best_model = model.fit(disp=False)
            best_params = (1, 1, 1, 0, 0, 0, 0)
        except:
            raise ValueError("All SARIMA configurations failed")

    if exog is not None:
        last_exog_value = exog.iloc[-1]
        future_exog = [last_exog_value] * forecast_steps
        forecast = best_model.forecast(steps=forecast_steps, exog=future_exog)
    else:
        forecast = best_model.forecast(steps=forecast_steps)

    if use_log:
        forecast = np.exp(forecast)

    param_string = f"({best_params[0]},{best_params[1]},{best_params[2]})({best_params[3]},{best_params[4]},{best_params[5]},{best_params[6]})"
    return pd.Series(forecast.values), param_string


def forecast_monthly_sarima(data, exog_col=None, forecast_steps=12):
    """Monthly SARIMA forecasting wrapper"""
    try:
        forecast, params = enhanced_sarima_forecast(data, exog_col, forecast_steps, use_log=False)
        last_date = pd.to_datetime(data['date'].iloc[-1])
        forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                                       periods=forecast_steps, freq='M')
        forecast_df = pd.DataFrame({'date': forecast_dates, 'forecast': forecast.values})
        return forecast_df, params
    except Exception as e:
        print(f"Monthly SARIMA failed: {e}")
        return None, None


# -----------------------------
# DB Helpers (with target_date)
# -----------------------------
def _table_exists(conn, db_name: str, tname: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.tables
        WHERE table_schema = :s AND table_name = :t
    """), {"s": db_name, "t": tname}).scalar())

def _column_exists(conn, db_name: str, tname: str, col: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = :s AND table_name = :t AND column_name = :c
    """), {"s": db_name, "t": tname, "c": col}).scalar())

def _index_exists(conn, db_name: str, tname: str, idx: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.statistics
        WHERE table_schema = :s AND table_name = :t AND index_name = :i
    """), {"s": db_name, "t": tname, "i": idx}).scalar())

def _ensure_table_and_indexes(engine, table_name: str, db_name: str):
    """combined_long_format(+target_date) 스키마 보장 - 개선된 버전"""
    with engine.begin() as conn:
        # 1) 테이블 생성 (최초 1회)
        if not _table_exists(conn, db_name, table_name):
            conn.execute(text(f"""
                CREATE TABLE `{table_name}` (
                  id BIGINT AUTO_INCREMENT PRIMARY KEY,
                  ticker         VARCHAR(32)  NOT NULL,
                  forecast_date  DATE         NOT NULL,
                  target_date    DATE         NOT NULL,
                  indicator      VARCHAR(128) NOT NULL,
                  frequency      VARCHAR(8)   NOT NULL,
                  value          DOUBLE       NULL,
                  exog_var       VARCHAR(64)  NULL,
                  params         VARCHAR(255) NULL,
                  valuation_time DATETIME     NOT NULL,
                  created_at     TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
                  INDEX `ix_ticker`        (ticker),
                  INDEX `ix_forecast_date` (forecast_date),
                  INDEX `ix_target_date`   (target_date),
                  INDEX `ix_indicator`     (indicator),
                  INDEX `ix_frequency`     (frequency)
                ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
            """))

        # 2) 기존의 잘못된 데이터 정리 (target_date가 NULL이거나 0000-00-00인 것들)
        print("기존 잘못된 데이터 정리 중...")
        try:
            cleanup_query = f"""
                DELETE FROM `{table_name}`
                WHERE target_date IS NULL
                   OR target_date = '0000-00-00'
                   OR target_date < '1900-01-01'
            """
            result = conn.execute(text(cleanup_query))
            if result.rowcount > 0:
                print(f"잘못된 데이터 {result.rowcount}건 삭제됨")
        except Exception as e:
            print(f"데이터 정리 중 오류: {e}")

        # 3) (구버전 테이블 대비) 누락 컬럼 보강
        for col, ddl in [
            ("forecast_date", "ADD COLUMN forecast_date DATE NOT NULL"),
            ("target_date",   "ADD COLUMN target_date DATE NOT NULL"),
            ("indicator",     "ADD COLUMN indicator VARCHAR(128) NOT NULL"),
            ("frequency",     "ADD COLUMN frequency VARCHAR(8) NOT NULL"),
            ("value",         "ADD COLUMN value DOUBLE NULL"),
            ("exog_var",      "ADD COLUMN exog_var VARCHAR(64) NULL"),
            ("params",        "ADD COLUMN params VARCHAR(255) NULL"),
            ("valuation_time","ADD COLUMN valuation_time DATETIME NOT NULL"),
        ]:
            if not _column_exists(conn, db_name, table_name, col):
                try:
                    conn.execute(text(f"ALTER TABLE `{table_name}` {ddl}"))
                except Exception as e:
                    print(f"컬럼 {col} 추가 중 오류: {e}")

        # 4) UNIQUE 인덱스 생성 (기존 인덱스 삭제 후 재생성)
        uniq = f"ux_{table_name}_keytime"
        try:
            # 기존 인덱스 삭제
            if _index_exists(conn, db_name, table_name, uniq):
                conn.execute(text(f"DROP INDEX `{uniq}` ON `{table_name}`"))
                print(f"기존 인덱스 {uniq} 삭제됨")

            # 새 인덱스 생성
            conn.execute(text(f"""
                CREATE UNIQUE INDEX `{uniq}`
                ON `{table_name}` (ticker, forecast_date, indicator, frequency, target_date, valuation_time)
            """))
            print(f"새 인덱스 {uniq} 생성됨")

        except Exception as e:
            print(f"인덱스 생성 중 오류: {e}")
            # 인덱스 없이 계속 진행


# -----------------------------
# Data Preparation
# -----------------------------
print("Loading base data...")
fundq_df = fetch_table_data(db_info, 'US_fundq')

if USE_EXOGENOUS:
    trade_df = fetch_table_data(db_info, 'us_trade_monthly_data_with_forecast')
else:
    trade_df = None

print("Collecting maximum revenue data from fundq...")
revenue_df = fundq_df[fundq_df['ticker'] == tic_name][['date', 'saleq']].copy()
revenue_df.rename(columns={'saleq': tic_name}, inplace=True)
revenue_df['date'] = pd.to_datetime(revenue_df['date'])
revenue_df = revenue_df.drop_duplicates(subset=['date']).sort_values('date')
revenue_df = revenue_df.dropna(subset=[tic_name])
print(f"Retrieved {len(revenue_df)} quarterly revenue records from {revenue_df['date'].min():%Y-%m} to {revenue_df['date'].max():%Y-%m}")

# TTM
revenue_df['TTM_Revenue'] = revenue_df[tic_name].rolling(window=4).sum()

endog_df = revenue_df.copy()
endog_df['quarter'] = endog_df['date'].dt.to_period('Q')
endog_df = endog_df.sort_values('date').groupby('quarter').last().reset_index(drop=True)
endog_df = endog_df[(endog_df['date'] >= st_date) & (endog_df['date'] <= end_date)]

# 외생 변수 준비 (분기 합계, yoy 등)
quarterly_sum_df = None
if USE_EXOGENOUS and trade_df is not None:
    print("Preparing exogenous data...")
    temp_df = trade_df[trade_df['hs_code_6d'] == hs_code][['date', 'expDlr']]
    target_export_df = temp_df.drop_duplicates(subset=['date'])
    target_export_df['date'] = pd.to_datetime(target_export_df['date'])
    target_export_df['quarter'] = target_export_df['date'].dt.to_period('Q')
    quarterly_sum_df = target_export_df.groupby('quarter')['expDlr'].sum().reset_index()
    quarterly_sum_df['quarter'] = quarterly_sum_df['quarter'].dt.to_timestamp()
    quarterly_sum_df['export_qoq_change'] = quarterly_sum_df['expDlr'].pct_change(periods=1)
    quarterly_sum_df['export_yoy_change'] = quarterly_sum_df['expDlr'].pct_change(periods=4)
    quarterly_sum_df['date_month'] = quarterly_sum_df['quarter'] + pd.offsets.QuarterEnd(0)
    quarterly_sum_df['date'] = pd.to_datetime(quarterly_sum_df['date_month'])
    quarterly_sum_df.set_index('date', inplace=True)
    quarterly_sum_df.drop(columns='quarter', inplace=True)
    quarterly_sum_df = quarterly_sum_df.reset_index()
    quarterly_sum_df = quarterly_sum_df.iloc[8:]  # 여유 버닝
    scaler = StandardScaler()
    quarterly_sum_df['exog_scaled'] = scaler.fit_transform(quarterly_sum_df[['export_yoy_change']])

# PSR 원천
print("Fetching PSR ratio data...")
ratios_timeseries = []
for ticker in tqdm([tic_name]):
    try:
        url = f"https://financialmodelingprep.com/api/v3/ratios/{ticker}?period=quarter&limit=5000&apikey={apikey}"
        response = requests.get(url)
        data = response.json()
        if isinstance(data, list):
            for entry in data:
                ratios_timeseries.append({
                    'ticker': ticker,
                    'date': entry.get('date'),
                    'PSR': entry.get('priceToSalesRatio'),
                    'PER': entry.get('priceEarningsRatio'),
                    'PBR': entry.get('priceToBookRatio'),
                    'ROE': entry.get('returnOnEquity'),
                    'ROA': entry.get('returnOnAssets')
                })
        else:
            print(f"Warning {ticker}: Unexpected data structure")
    except Exception as e:
        print(f"Error {ticker}: {e}")

if len(ratios_timeseries) == 0:
    raise ValueError("No ratio data retrieved from FMP API")

df_ratios = pd.DataFrame(ratios_timeseries)
df_ratios.sort_values(by=['ticker', 'date'], inplace=True)
print(f"Retrieved {len(df_ratios)} quarterly ratio records")

# PSR 분기 정렬
print("Processing ratio data...")
endog_ratio_df = df_ratios[(df_ratios['ticker'] == tic_name)][['date', 'PSR']].copy()
endog_ratio_df['date'] = pd.to_datetime(endog_ratio_df['date'])
endog_ratio_df = endog_ratio_df.dropna(subset=['PSR'])

def enhanced_quarter_alignment(df):
    df = df.copy().sort_values('date').reset_index(drop=True)
    def flexible_quarter_end(date):
        y, m = date.year, date.month
        if m <= 3:  return pd.Timestamp(y, 3, 31)
        if m <= 6:  return pd.Timestamp(y, 6, 30)
        if m <= 9:  return pd.Timestamp(y, 9, 30)
        return pd.Timestamp(y, 12, 31)
    df['quarter_end'] = df['date'].apply(flexible_quarter_end)
    q = df.groupby('quarter_end').agg({'PSR': 'last', 'date': 'last'}).reset_index()
    q['date'] = q['quarter_end']
    return q[['date','PSR']]

endog_ratio_df = enhanced_quarter_alignment(endog_ratio_df)
print(f"After enhanced alignment: {len(endog_ratio_df)} quarterly PSR records")

# -----------------------------
# Build datasets (Revenue / PSR)
# -----------------------------
print("Creating revenue forecasting dataset...")
if USE_EXOGENOUS and quarterly_sum_df is not None:
    revenue_start, revenue_end = endog_df['date'].min(), endog_df['date'].max()
    exog_start, exog_end = quarterly_sum_df['date'].min(), quarterly_sum_df['date'].max()
    merge_start, merge_end = max(revenue_start, exog_start), min(revenue_end, exog_end)
    print(f"Revenue merge period: {merge_start:%Y-%m-%d} to {merge_end:%Y-%m-%d}")
    merged_revenue = merge_endog_exog_data(
        endog_df=endog_df,
        exog_df=quarterly_sum_df,
        endog_col=tic_name,
        exog_col='export_yoy_change',
        start_date=merge_start.strftime('%Y-%m-%d'),
        end_date=merge_end.strftime('%Y-%m-%d')
    )
    merged_revenue = merged_revenue.drop_duplicates(subset=['date', 'endog_var'], keep='first').reset_index(drop=True)
    print(f"Revenue dataset with exog: {len(merged_revenue)} records")
else:
    merged_revenue = endog_df.copy()
    merged_revenue.rename(columns={tic_name: 'endog_var'}, inplace=True)
    merged_revenue['exog_var'] = None
    print(f"Revenue dataset without exog: {len(merged_revenue)} records")

print("Creating PSR forecasting dataset...")
if USE_EXOGENOUS and quarterly_sum_df is not None:
    exog_for_psr = quarterly_sum_df.copy()
    exog_for_psr['date'] = pd.to_datetime(exog_for_psr['date'])
    merged_ratio_exog_full = pd.merge(
        endog_ratio_df,
        exog_for_psr[['date', 'exog_scaled']],
        on='date', how='outer'
    ).sort_values('date')
    print(f"Full merge result: {len(merged_ratio_exog_full)} records")
    merged_ratio_exog_full['PSR'] = merged_ratio_exog_full['PSR'].interpolate(method='linear')
    merged_ratio_exog_full['exog_scaled'] = (merged_ratio_exog_full['exog_scaled']
                                             .fillna(method='ffill').fillna(method='bfill'))
    start_filter, end_filter = '2010-01-01', '2026-12-31'
    merged_ratio_exog = merged_ratio_exog_full[
        (merged_ratio_exog_full['date'] >= start_filter) &
        (merged_ratio_exog_full['date'] <= end_filter)
    ][['date','PSR','exog_scaled']].copy()
    merged_ratio_exog = merged_ratio_exog.dropna()
    merged_ratio_exog.rename(columns={'PSR':'endog_var','exog_scaled':'exog_var'}, inplace=True)
    print(f"Final PSR dataset with exog: {len(merged_ratio_exog)} records")
else:
    merged_ratio_exog = endog_ratio_df.copy()
    merged_ratio_exog = merged_ratio_exog[merged_ratio_exog['date'] >= '2010-01-01'].copy()
    merged_ratio_exog.rename(columns={'PSR':'endog_var'}, inplace=True)
    merged_ratio_exog['exog_var'] = None
    print(f"PSR dataset without exog: {len(merged_ratio_exog)} records")

# -----------------------------
# Forecasting (Revenue / PSR)
# -----------------------------
print("Forecasting revenue with SARIMA...")
revenue_forecasts_with_exog = None
revenue_params_with_exog = None
revenue_forecasts_without_exog = None
revenue_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in merged_revenue.columns and merged_revenue['exog_var'].notna().any():
    try:
        print("Forecasting revenue with exogenous variables...")
        revenue_forecasts_with_exog, revenue_params_with_exog = enhanced_sarima_forecast(
            merged_revenue, exog_col='exog_var', forecast_steps=4, use_log=True
        )
        print(f"Revenue with exog - Best params: {revenue_params_with_exog}")
    except Exception as e:
        print(f"Revenue exogenous forecasting failed: {str(e)}")

try:
    print("Forecasting revenue without exogenous variables...")
    revenue_forecasts_without_exog, revenue_params_without_exog = enhanced_sarima_forecast(
        merged_revenue, exog_col=None, forecast_steps=4, use_log=False
    )
    print(f"Revenue without exog - Best params: {revenue_params_without_exog}")
except Exception as e:
    print(f"Revenue forecasting failed: {str(e)}")
    last_revenue = merged_revenue['endog_var'].iloc[-1]
    revenue_forecasts_without_exog = pd.Series([last_revenue * 1.05] * 4)
    revenue_params_without_exog = "fallback"

print("Forecasting quarterly PSR with SARIMA...")
ratio_forecasts_with_exog = None
ratio_params_with_exog = None
ratio_forecasts_without_exog = None
ratio_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in merged_ratio_exog.columns and merged_ratio_exog['exog_var'].notna().any():
    try:
        print("Forecasting PSR with exogenous variables...")
        ratio_forecasts_with_exog, ratio_params_with_exog = enhanced_sarima_forecast(
            merged_ratio_exog, exog_col='exog_var', forecast_steps=4, use_log=False
        )
        print(f"PSR with exog - Best params: {ratio_params_with_exog}")
    except Exception as e:
        print(f"PSR exogenous forecasting failed: {str(e)}")

try:
    print("Forecasting PSR without exogenous variables...")
    ratio_forecasts_without_exog, ratio_params_without_exog = enhanced_sarima_forecast(
        merged_ratio_exog, exog_col=None, forecast_steps=4, use_log=False
    )
    print(f"PSR without exog - Best params: {ratio_params_without_exog}")
except Exception as e:
    print(f"PSR forecasting failed: {str(e)}")
    last_psr = merged_ratio_exog['endog_var'].iloc[-1]
    ratio_forecasts_without_exog = pd.Series([last_psr] * 4)
    ratio_params_without_exog = "fallback"

# Monthly dataset (for ML & monthly SARIMA)
print("Preparing monthly PSR data for ML forecasting...")
from_date = '2015-01-01'
to_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

try:
    df_market = get_market_cap(tic_name, apikey, from_date, to_date)
    print(f"Retrieved {len(df_market)} daily market cap records")
    df_market = df_market.sort_values('date')
    revenue_for_merge = revenue_df[['date', 'TTM_Revenue']].dropna()

    df_daily_psr = pd.merge_asof(
        df_market, revenue_for_merge, left_on='date', right_on='date', direction='backward'
    )
    df_daily_psr['PSR'] = df_daily_psr['marketCap'] / df_daily_psr['TTM_Revenue']
    df_daily_psr = df_daily_psr.dropna(subset=['PSR'])
    df_daily_psr['date'] = pd.to_datetime(df_daily_psr['date'])

    monthly_psr_data = df_daily_psr.set_index('date').resample('M').last().reset_index()
    monthly_psr_data = monthly_psr_data.dropna(subset=['PSR'])
    print(f"Monthly PSR from actual market data: {len(monthly_psr_data)} records")

    if USE_EXOGENOUS and quarterly_sum_df is not None:
        quarterly_export_for_monthly = quarterly_sum_df.copy()
        quarterly_export_for_monthly['date'] = pd.to_datetime(quarterly_export_for_monthly['date'])
        monthly_export_data = []
        for _, row in quarterly_export_for_monthly.iterrows():
            quarter_date = row['date']; quarter = quarter_date.quarter; year = quarter_date.year
            months = [1,2,3] if quarter == 1 else [4,5,6] if quarter == 2 else [7,8,9] if quarter == 3 else [10,11,12]
            for m in months:
                month_end = pd.Timestamp(year, m, 1) + pd.offsets.MonthEnd(0)
                monthly_export_data.append({'date': month_end, 'export_yoy_change': row['export_yoy_change']})
        monthly_export_df = pd.DataFrame(monthly_export_data)
        monthly_merged = pd.merge(
            monthly_psr_data[['date','PSR']], monthly_export_df, on='date', how='left'
        )
        scaler = StandardScaler()
        monthly_merged['exog_scaled'] = scaler.fit_transform(monthly_merged[['export_yoy_change']])
        monthly_merged = monthly_merged.dropna()
        monthly_merged.rename(columns={'PSR':'endog_var','exog_scaled':'exog_var'}, inplace=True)
        print(f"Monthly PSR dataset with exogenous: {len(monthly_merged)} records")
    else:
        monthly_merged = monthly_psr_data[['date','PSR']].copy()
        monthly_merged.rename(columns={'PSR':'endog_var'}, inplace=True)
        monthly_merged['exog_var'] = None
        print(f"Monthly PSR dataset without exogenous: {len(monthly_merged)} records")

except Exception as e:
    print(f"Could not create monthly PSR data: {e}")
    monthly_merged = merged_ratio_exog.copy()

# Monthly SARIMA
print("Forecasting monthly PSR with SARIMA...")
ratio_monthly_forecasts_with_exog = None
ratio_monthly_params_with_exog = None
ratio_monthly_forecasts_without_exog = None
ratio_monthly_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in monthly_merged.columns and monthly_merged['exog_var'].notna().any():
    try:
        ratio_monthly_forecasts_with_exog, ratio_monthly_params_with_exog = forecast_monthly_sarima(
            monthly_merged, exog_col='exog_var', forecast_steps=12
        )
        print(f"Monthly PSR with exog - Best params: {ratio_monthly_params_with_exog}")
    except Exception as e:
        print(f"Monthly PSR exogenous forecasting failed: {str(e)}")

try:
    ratio_monthly_forecasts_without_exog, ratio_monthly_params_without_exog = forecast_monthly_sarima(
        monthly_merged, exog_col=None, forecast_steps=12
    )
    print(f"Monthly PSR without exog - Best params: {ratio_monthly_params_without_exog}")
except Exception as e:
    print(f"Monthly PSR forecasting failed: {str(e)}")
    last_date = pd.to_datetime(monthly_merged['date'].iloc[-1])
    last_value = monthly_merged['endog_var'].iloc[-1]
    forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='M')
    ratio_monthly_forecasts_without_exog = pd.DataFrame({'date': forecast_dates, 'forecast': [last_value] * 12})
    ratio_monthly_params_without_exog = "fallback"

# LSTM & Prophet
print("Forecasting PSR with LSTM and Prophet...")
print("Applying LSTM forecasting...")
lstm_forecast = forecast_psr_with_lstm(monthly_merged, forecast_steps=12)
if lstm_forecast is not None:
    print(f"LSTM forecast completed: {len(lstm_forecast)} months")
else:
    print("LSTM forecast failed, creating fallback")
    last_date = pd.to_datetime(monthly_merged['date'].iloc[-1])
    last_values = monthly_merged['endog_var'].tail(3).mean()
    forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='M')
    lstm_forecast = pd.DataFrame({'date': forecast_dates, 'forecasted_PSR': [last_values] * 12})

print("Applying Prophet forecasting...")
prophet_forecast = forecast_psr_with_prophet(monthly_merged, forecast_steps=12)
if prophet_forecast is not None:
    print(f"Prophet forecast completed: {len(prophet_forecast)} months")
else:
    print("Prophet forecast failed, creating fallback")
    last_date = pd.to_datetime(monthly_merged['date'].iloc[-1])
    last_values = monthly_merged['endog_var'].tail(3).mean()
    forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='M')
    prophet_forecast = pd.DataFrame({'date': forecast_dates, 'forecasted_PSR': [last_values] * 12})

# -----------------------------
# Valuation Assembly
# -----------------------------
print("Compiling final results with forward-looking valuation logic...")

# Forward revenue (next 4Q)
if revenue_forecasts_with_exog is not None:
    forward_revenue_with_exog = revenue_forecasts_with_exog.sum()
    revenue_forecasts_with_exog = revenue_forecasts_with_exog.rename('revenue_with_exog')
else:
    forward_revenue_with_exog = None

forward_revenue_without_exog = revenue_forecasts_without_exog.sum()
revenue_forecasts_without_exog = revenue_forecasts_without_exog.rename('revenue_without_exog')

print("Forecasted Forward Revenue (Next 4 Quarters):")
if forward_revenue_with_exog is not None:
    print(f"   With exogenous variables: ${forward_revenue_with_exog/1e6:.1f}M")
print(f"   Without exogenous variables: ${forward_revenue_without_exog/1e6:.1f}M")

# Long-term (Quarterly)
print("Creating long-term valuation forecasts (Quarterly)...")
longterm_series_data, longterm_names = [], []
if ratio_forecasts_with_exog is not None:
    longterm_series_data.extend([ratio_forecasts_with_exog, revenue_forecasts_with_exog])
    longterm_names.extend(['PSR_quarter_with_exog', 'revenue_with_exog'])
longterm_series_data.extend([ratio_forecasts_without_exog, revenue_forecasts_without_exog])
longterm_names.extend(['PSR_quarter_without_exog', 'revenue_without_exog'])

longterm_value_forecasts = pd.concat(longterm_series_data, axis=1, keys=longterm_names)

if forward_revenue_with_exog is not None:
    longterm_value_forecasts['forward_revenue_with_exog'] = forward_revenue_with_exog
longterm_value_forecasts['forward_revenue_without_exog'] = forward_revenue_without_exog

# EV using quarterly PSR × forward revenue
if ratio_forecasts_with_exog is not None:
    longterm_value_forecasts['value_with_exog'] = (
        longterm_value_forecasts['PSR_quarter_with_exog'] * longterm_value_forecasts['forward_revenue_with_exog']
    )
longterm_value_forecasts['value_without_exog'] = (
    longterm_value_forecasts['PSR_quarter_without_exog'] * longterm_value_forecasts['forward_revenue_without_exog']
)

# ----- target_date for Quarterly -----
last_q_obs = max(pd.to_datetime(endog_ratio_df['date']).max(),
                 pd.to_datetime(endog_df['date']).max())
q_dates = pd.date_range(start=last_q_obs + pd.offsets.QuarterEnd(1),
                        periods=len(longterm_value_forecasts), freq='Q')
longterm_value_forecasts = longterm_value_forecasts.reset_index(drop=True)
longterm_value_forecasts['target_date'] = q_dates   # ✅ 핵심
longterm_value_forecasts['frequency'] = 'Q'
longterm_value_forecasts['ticker'] = tic_name
longterm_value_forecasts['forecast_date'] = today_date

# Mid-term (Monthly SARIMA)
print("Creating mid-term valuation forecasts (Monthly SARIMA)...")
midterm_series_data, midterm_names = [], []
if ratio_monthly_forecasts_with_exog is not None:
    midterm_series_data.append(ratio_monthly_forecasts_with_exog['forecast'])
    midterm_names.append('PSR_monthly_sarima_with_exog')
midterm_series_data.append(ratio_monthly_forecasts_without_exog['forecast'])
midterm_names.append('PSR_monthly_sarima_without_exog')

midterm_value_forecasts = pd.concat(midterm_series_data, axis=1, keys=midterm_names)

if ratio_monthly_forecasts_with_exog is not None:
    midterm_value_forecasts['forward_revenue_with_exog'] = forward_revenue_with_exog
midterm_value_forecasts['forward_revenue_without_exog'] = forward_revenue_without_exog

if ratio_monthly_forecasts_with_exog is not None:
    src_dates = ratio_monthly_forecasts_with_exog['date'].values
else:
    src_dates = ratio_monthly_forecasts_without_exog['date'].values

midterm_value_forecasts = midterm_value_forecasts.reset_index(drop=True)
midterm_value_forecasts['target_date'] = src_dates  # ✅ 핵심
midterm_value_forecasts['frequency'] = 'M'
midterm_value_forecasts['ticker'] = tic_name
midterm_value_forecasts['forecast_date'] = today_date

# EV using monthly SARIMA PSR × forward revenue
if ratio_monthly_forecasts_with_exog is not None:
    midterm_value_forecasts['value_monthly_sarima_with_exog'] = (
        midterm_value_forecasts['PSR_monthly_sarima_with_exog'] *
        midterm_value_forecasts['forward_revenue_with_exog']
    )
midterm_value_forecasts['value_monthly_sarima_without_exog'] = (
    midterm_value_forecasts['PSR_monthly_sarima_without_exog'] *
    midterm_value_forecasts['forward_revenue_without_exog']
)

# Short-term (Monthly ML: LSTM & Prophet)
print("Creating short-term valuation forecasts (Monthly ML)...")
lstm_forecast_clean = lstm_forecast.copy()
prophet_forecast_clean = prophet_forecast.copy()
lstm_forecast_clean = lstm_forecast_clean.drop_duplicates(subset=['date']).sort_values('date')
prophet_forecast_clean = prophet_forecast_clean.drop_duplicates(subset=['date']).sort_values('date')

shortterm_forecasts = pd.merge(
    lstm_forecast_clean[['date','forecasted_PSR']],
    prophet_forecast_clean[['date','forecasted_PSR']],
    on='date', how='inner', suffixes=('_lstm','_prophet')
)

# target_date 로 승격
shortterm_forecasts = shortterm_forecasts.rename(columns={'date':'target_date'})  # ✅ 핵심

if forward_revenue_with_exog is not None:
    shortterm_forecasts['forward_revenue_with_exog'] = forward_revenue_with_exog
shortterm_forecasts['forward_revenue_without_exog'] = forward_revenue_without_exog

# EV using ML PSR × forward revenue
if forward_revenue_with_exog is not None:
    shortterm_forecasts['lstm_value_with_exog'] = shortterm_forecasts['forecasted_PSR_lstm'] * forward_revenue_with_exog
    shortterm_forecasts['prophet_value_with_exog'] = shortterm_forecasts['forecasted_PSR_prophet'] * forward_revenue_with_exog
shortterm_forecasts['lstm_value_without_exog'] = shortterm_forecasts['forecasted_PSR_lstm'] * forward_revenue_without_exog
shortterm_forecasts['prophet_value_without_exog'] = shortterm_forecasts['forecasted_PSR_prophet'] * forward_revenue_without_exog

shortterm_forecasts['frequency'] = 'M'
shortterm_forecasts['ticker'] = tic_name
shortterm_forecasts['forecast_date'] = today_date

# -----------------------------
# Melt & Combine (with target_date)
# -----------------------------
print("Melting and combining results...")

def melt_forecast_df_enhanced(df, freq, params_dict=None):
    melted = df.melt(
        id_vars=['frequency','ticker','forecast_date','target_date'],  # ✅ target_date 포함
        var_name='indicator',
        value_name='value'
    )
    melted['exog_var'] = hs_code if USE_EXOGENOUS else None
    melted['params'] = None
    if params_dict:
        for indicator, param in params_dict.items():
            if param:
                melted.loc[melted['indicator'] == indicator, 'params'] = param
    return melted

# 파라미터 사전
longterm_params = {}
if revenue_params_with_exog:
    longterm_params['revenue_with_exog'] = revenue_params_with_exog
if revenue_params_without_exog:
    longterm_params['revenue_without_exog'] = revenue_params_without_exog
if ratio_params_with_exog:
    longterm_params['PSR_quarter_with_exog'] = ratio_params_with_exog
if ratio_params_without_exog:
    longterm_params['PSR_quarter_without_exog'] = ratio_params_without_exog

midterm_params = {}
if ratio_monthly_params_with_exog:
    midterm_params['PSR_monthly_sarima_with_exog'] = ratio_monthly_params_with_exog
if ratio_monthly_params_without_exog:
    midterm_params['PSR_monthly_sarima_without_exog'] = ratio_monthly_params_without_exog

# Melt
longterm_melted  = melt_forecast_df_enhanced(longterm_value_forecasts, 'Q', longterm_params)
midterm_melted   = melt_forecast_df_enhanced(midterm_value_forecasts,  'M', midterm_params)
shortterm_melted = melt_forecast_df_enhanced(shortterm_forecasts,      'M')

# ML 모델 태깅
lstm_mask    = shortterm_melted['indicator'].str.contains('lstm',    na=False)
prophet_mask = shortterm_melted['indicator'].str.contains('prophet', na=False)
shortterm_melted.loc[lstm_mask,    'params'] = 'LSTM'
shortterm_melted.loc[prophet_mask, 'params'] = 'Prophet'

# 합치기
combined_long_format = pd.concat([longterm_melted, midterm_melted, shortterm_melted],
                                 axis=0, ignore_index=True)

print("Forward-looking valuation completed successfully!")
print(f"Results summary:")
print(f"   Total records: {len(combined_long_format)}")
print(f"   Exogenous variable used: {'Yes (' + hs_code + ')' if USE_EXOGENOUS else 'No'}")
print(f"   Frequencies: {combined_long_format['frequency'].unique()}")
print(f"   Unique indicators: {len(combined_long_format['indicator'].unique())}")
print(f"   Valuation method: Forward PSR × Forward Revenue (Next 4Q)")

# 샘플 출력
print("\nSample forward-looking enterprise valuations:")
value_indicators = combined_long_format[combined_long_format['indicator'].str.contains('value', na=False)]
if len(value_indicators) > 0:
    sample_valuations = value_indicators.head(10)[['indicator','value','params','frequency']]
    for _, row in sample_valuations.iterrows():
        value_billions = row['value'] / 1e9 if pd.notna(row['value']) else 0
        print(f"   {row['indicator']}: ${value_billions:.1f}B ({row['params']}, {row['frequency']})")

print("\nSample results:")
print(combined_long_format.head(10))

def save_valuation_results_to_db(db_info, combined_long_format):
    """
    최종 해결책: 모든 중복 문제를 해결한 안전한 저장 함수
    """
    try:
        from sqlalchemy import create_engine, text
        from datetime import datetime, timedelta
        import pandas as pd
        import uuid

        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        engine = create_engine(connection_string)
        table_name = 'US_company_valuation_result'

        # ==============================================
        # 1단계: 데이터 전처리 및 중복 제거
        # ==============================================
        print("=== 데이터 전처리 시작 ===")
        df_to_save = combined_long_format.copy()

        # 날짜 컬럼 정리
        if 'target_date' in df_to_save.columns:
            df_to_save['target_date'] = pd.to_datetime(df_to_save['target_date'])
        if 'forecast_date' in df_to_save.columns:
            df_to_save['forecast_date'] = pd.to_datetime(df_to_save['forecast_date'])

        # NaN 처리
        df_to_save = df_to_save.where(pd.notnull(df_to_save), None)

        print(f"전처리 전 총 레코드: {len(df_to_save)}건")

        # 중복 제거 (UNIQUE 키 기준으로 - valuation_time 제외)
        unique_cols = ['ticker', 'forecast_date', 'indicator', 'frequency', 'target_date']
        df_to_save = df_to_save.drop_duplicates(subset=unique_cols, keep='last')
        print(f"중복 제거 후 레코드: {len(df_to_save)}건")

        # valuation_time을 각 행마다 고유하게 설정 (마이크로초 단위로 차별화)
        base_time = datetime.now()
        df_to_save = df_to_save.reset_index(drop=True)
        df_to_save['valuation_time'] = [base_time + timedelta(microseconds=i) for i in range(len(df_to_save))]

        print(f"최종 처리할 데이터: {len(df_to_save)}건")

        # ==============================================
        # 2단계: 테이블 스키마 확인 및 준비
        # ==============================================
        print("=== 테이블 스키마 확인 ===")
        _ensure_table_and_indexes(engine, table_name, db_info['database'])

        ticker = df_to_save['ticker'].iloc[0]
        forecast_date = df_to_save['forecast_date'].iloc[0]

        # ==============================================
        # 3단계: 강력한 기존 데이터 삭제
        # ==============================================
        print("=== 기존 데이터 완전 삭제 ===")
        with engine.begin() as conn:
            # 여러 방법으로 기존 데이터 삭제 시도
            delete_queries = [
                # 방법 1: 정확한 날짜 매칭
                f"""DELETE FROM `{table_name}`
                   WHERE ticker = :ticker AND forecast_date = :forecast_date""",

                # 방법 2: 날짜 범위로 삭제
                f"""DELETE FROM `{table_name}`
                   WHERE ticker = :ticker
                   AND DATE(forecast_date) = DATE(:forecast_date)""",

                # 방법 3: 더 넓은 범위로 삭제
                f"""DELETE FROM `{table_name}`
                   WHERE ticker = :ticker
                   AND forecast_date >= :start_date
                   AND forecast_date < :end_date"""
            ]

            start_date = pd.to_datetime(forecast_date).replace(hour=0, minute=0, second=0, microsecond=0)
            end_date = start_date + pd.Timedelta(days=1)

            total_deleted = 0
            for i, delete_query in enumerate(delete_queries, 1):
                try:
                    result = conn.execute(text(delete_query), {
                        'ticker': ticker,
                        'forecast_date': forecast_date,
                        'start_date': start_date,
                        'end_date': end_date
                    })
                    deleted_count = result.rowcount
                    total_deleted += deleted_count
                    print(f"삭제 방법 {i}: {deleted_count}건 삭제")

                    if deleted_count > 0:
                        break  # 성공적으로 삭제했으면 다음 방법은 시도하지 않음

                except Exception as del_error:
                    print(f"삭제 방법 {i} 실패: {del_error}")
                    continue

            # 삭제 후 확인
            check_query = text(f"""
                SELECT COUNT(*) FROM `{table_name}`
                WHERE ticker = :ticker
                AND DATE(forecast_date) = DATE(:forecast_date)
            """)
            remaining = conn.execute(check_query, {
                'ticker': ticker,
                'forecast_date': forecast_date
            }).scalar()

            print(f"총 삭제된 데이터: {total_deleted}건, 남은 데이터: {remaining}건")

            # ==============================================
            # 4단계: 안전한 데이터 삽입 (IGNORE 방식)
            # ==============================================
            print("=== 안전한 데이터 삽입 ===")

            # INSERT IGNORE를 사용하여 중복 시 무시
            insert_query = text(f"""
                INSERT IGNORE INTO `{table_name}`
                (frequency, ticker, forecast_date, target_date, indicator, value, exog_var, params, valuation_time)
                VALUES (:frequency, :ticker, :forecast_date, :target_date, :indicator, :value, :exog_var, :params, :valuation_time)
            """)

            successful_inserts = 0
            failed_inserts = 0

            # 개별 행 처리 (가장 안전한 방법)
            for idx, row in df_to_save.iterrows():
                try:
                    result = conn.execute(insert_query, {
                        'frequency': row['frequency'],
                        'ticker': row['ticker'],
                        'forecast_date': row['forecast_date'],
                        'target_date': row['target_date'],
                        'indicator': row['indicator'],
                        'value': row['value'],
                        'exog_var': row['exog_var'],
                        'params': row['params'],
                        'valuation_time': row['valuation_time']
                    })

                    if result.rowcount > 0:
                        successful_inserts += 1
                    else:
                        failed_inserts += 1
                        print(f"중복으로 무시됨: {row['indicator']} ({row['target_date']})")

                except Exception as insert_error:
                    failed_inserts += 1
                    print(f"삽입 실패: {row['indicator']} - {insert_error}")

                # 진행 상황 표시 (100건마다)
                if (idx + 1) % 100 == 0:
                    print(f"진행: {idx + 1}/{len(df_to_save)}건 처리 완료")

            print(f"삽입 완료 - 성공: {successful_inserts}건, 실패/중복: {failed_inserts}건")

        # ==============================================
        # 5단계: 결과 확인
        # ==============================================
        print("=== 저장 결과 확인 ===")
        with engine.connect() as conn:
            # 총 레코드 수 확인
            total_count_query = text(f"SELECT COUNT(*) FROM `{table_name}` WHERE ticker = :ticker")
            total_count = conn.execute(total_count_query, {'ticker': ticker}).scalar()

            # 오늘 저장된 레코드 수 확인
            today_count_query = text(f"""
                SELECT COUNT(*) FROM `{table_name}`
                WHERE ticker = :ticker AND DATE(forecast_date) = DATE(:forecast_date)
            """)
            today_count = conn.execute(today_count_query, {
                'ticker': ticker,
                'forecast_date': forecast_date
            }).scalar()

            print(f"저장 완료:")
            print(f"  - 오늘 저장된 레코드: {today_count}건")
            print(f"  - {ticker} 전체 레코드: {total_count}건")

        engine.dispose()

        if successful_inserts > 0:
            print("✅ 데이터베이스 저장 성공!")
            return True
        else:
            print("❌ 새로 저장된 데이터가 없습니다.")
            return False

    except Exception as e:
        print(f"❌ 저장 중 치명적 오류 발생: {str(e)}")
        print(f"오류 타입: {type(e).__name__}")

        # 백업 저장
        try:
            backup_filename = f"valuation_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            combined_long_format.to_csv(backup_filename, index=False)
            print(f"📁 백업 파일 저장됨: {backup_filename}")
        except Exception as backup_error:
            print(f"백업 저장도 실패: {backup_error}")

        return False


# ==============================================
# 추가 도구: 데이터베이스 상태 확인 함수
# ==============================================
def check_database_status(db_info, ticker=tic_name):
    """데이터베이스 현재 상태 확인"""
    try:
        from sqlalchemy import create_engine, text

        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        engine = create_engine(connection_string)
        table_name = 'US_company_valuation_result'

        with engine.connect() as conn:
            # 테이블 존재 확인
            table_check = conn.execute(text("""
                SELECT COUNT(*) FROM information_schema.tables
                WHERE table_schema = :db AND table_name = :table
            """), {'db': db_info['database'], 'table': table_name}).scalar()

            if table_check == 0:
                print(f"⚠️  테이블 '{table_name}'이 존재하지 않습니다.")
                return

            # 전체 데이터 현황
            total_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar()

            # 특정 ticker 데이터 현황
            ticker_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}` WHERE ticker = :ticker"),
                                      {'ticker': ticker}).scalar()

            # 최근 저장된 데이터
            recent_data = conn.execute(text(f"""
                SELECT forecast_date, COUNT(*) as count
                FROM `{table_name}`
                WHERE ticker = :ticker
                GROUP BY forecast_date
                ORDER BY forecast_date DESC
                LIMIT 5
            """), {'ticker': ticker}).fetchall()

            print(f"📊 데이터베이스 현황:")
            print(f"  - 전체 레코드: {total_count:,}건")
            print(f"  - {ticker} 레코드: {ticker_count:,}건")
            print(f"  - 최근 저장 이력:")
            for row in recent_data:
                print(f"    {row[0]}: {row[1]}건")

        engine.dispose()

    except Exception as e:
        print(f"상태 확인 실패: {e}")


# ==============================================
# 응급 복구 함수 (필요시 사용)
# ==============================================
def emergency_cleanup(db_info, ticker=tic_name, date_str='2025-09-04'):
    """응급 상황시 특정 데이터 완전 삭제"""
    try:
        from sqlalchemy import create_engine, text

        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        engine = create_engine(connection_string)
        table_name = 'US_company_valuation_result'

        with engine.begin() as conn:
            result = conn.execute(text(f"""
                DELETE FROM `{table_name}`
                WHERE ticker = :ticker
                AND DATE(forecast_date) = :date
            """), {'ticker': ticker, 'date': date_str})

            print(f"🗑️  응급 삭제 완료: {result.rowcount}건 삭제됨")

        engine.dispose()
        return True

    except Exception as e:
        print(f"응급 삭제 실패: {e}")
        return False

# -----------------------------
# Save to DB (with target_date key)
# -----------------------------
print("\nSaving results to database...")
save_success = save_valuation_results_to_db(db_info, combined_long_format)

if save_success:
    print("All operations completed successfully!")
else:
    print("Forecasting completed, but database save failed. Data is still available in memory.")

# -----------------------------
# Final Summary
# -----------------------------
print(f"\nFinal Summary:")
print(f"Ticker: {tic_name}")
print(f"Forecast Date: {today_date.strftime('%Y-%m-%d')}")
print(f"Exogenous Variable: {hs_code if USE_EXOGENOUS else 'None'}")
print(f"Forward Revenue (4Q): ${forward_revenue_without_exog/1e6:.1f}M")
print(f"Total Result Records: {len(combined_long_format)}")
print(f"Forecast Methods: SARIMA (Q/M), LSTM (M), Prophet (M)")
print(f"Valuation Logic: Future PSR × Future Revenue")

print("\nFinal Results Structure:")
print("Columns:", list(combined_long_format.columns))
print("\nIndicator Types:")
unique_indicators = combined_long_format['indicator'].unique()
for indicator in sorted(unique_indicators):
    count = len(combined_long_format[combined_long_format['indicator'] == indicator])
    print(f"  {indicator}: {count} records")

print("\nStock valuation forecasting completed successfully!")
print("Data saved to: US_company_valuation_result")


Loading base data...
✅ 'US_fundq' 테이블에서 1334117건의 데이터를 가져왔습니다.
✅ 'us_trade_monthly_data_with_forecast' 테이블에서 74279건의 데이터를 가져왔습니다.
Retrieved 143 quarterly revenue records from 2000-01 to 2024-12
Preparing exogenous data...
Fetching PSR ratio data...


100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


Retrieved 51 quarterly ratio records
Processing ratio data...
After enhanced alignment: 51 quarterly PSR records
Creating revenue forecasting dataset...
Revenue merge period: 2015-03-31 to 2024-12-31
Revenue dataset with exog: 40 records
Creating PSR forecasting dataset...
Full merge result: 56 records
Final PSR dataset with exog: 56 records
Forecasting revenue with SARIMA...
Forecasting revenue with exogenous variables...
Revenue with exog - Best params: (1,0,0)(0,0,0,4)
Forecasting revenue without exogenous variables...
Revenue without exog - Best params: (0,1,2)(0,1,1,4)
Forecasting quarterly PSR with SARIMA...
Forecasting PSR with exogenous variables...
PSR with exog - Best params: (0,1,2)(0,1,1,4)
Forecasting PSR without exogenous variables...
PSR without exog - Best params: (0,1,2)(0,1,1,4)
Preparing monthly PSR data for ML forecasting...
Retrieved 1307 daily market cap records
Monthly PSR from actual market data: 64 records
Monthly PSR dataset with exogenous: 64 records
Forecast

DEBUG:cmdstanpy:cmd: where.exe tbb.dll
cwd: None
DEBUG:cmdstanpy:TBB already found in load path
DEBUG:cmdstanpy:input tempfile: C:\Users\82108\AppData\Local\Temp\tmpsmrc37y7\yiiwtyeb.json
DEBUG:cmdstanpy:input tempfile: C:\Users\82108\AppData\Local\Temp\tmpsmrc37y7\y35586yv.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['C:\\Users\\82108\\AppData\\Local\\Programs\\Python\\Python39\\Lib\\site-packages\\prophet\\stan_model\\prophet_model.bin', 'random', 'seed=76432', 'data', 'file=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpsmrc37y7\\yiiwtyeb.json', 'init=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpsmrc37y7\\y35586yv.json', 'output', 'file=C:\\Users\\82108\\AppData\\Local\\Temp\\tmpsmrc37y7\\prophet_modelhn8am9l5\\prophet_model-20250905002731.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
00:27:31 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing


LSTM forecast completed: 12 months
Applying Prophet forecasting...


00:27:31 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Prophet forecast completed: 12 months
Compiling final results with forward-looking valuation logic...
Forecasted Forward Revenue (Next 4 Quarters):
   With exogenous variables: $0.0M
   Without exogenous variables: $0.0M
Creating long-term valuation forecasts (Quarterly)...
Creating mid-term valuation forecasts (Monthly SARIMA)...
Creating short-term valuation forecasts (Monthly ML)...
Melting and combining results...
Forward-looking valuation completed successfully!
Results summary:
   Total records: 200
   Exogenous variable used: Yes (851762)
   Frequencies: ['Q' 'M']
   Unique indicators: 18
   Valuation method: Forward PSR × Forward Revenue (Next 4Q)

Sample forward-looking enterprise valuations:
   value_with_exog: $0.0B (None, Q)
   value_with_exog: $0.0B (None, Q)
   value_with_exog: $0.0B (None, Q)
   value_with_exog: $0.0B (None, Q)
   value_without_exog: $0.0B (None, Q)
   value_without_exog: $0.0B (None, Q)
   value_without_exog: $0.0B (None, Q)
   value_without_exog: $0.0B